# IRRM-CODEC: multi-chain forward analysis

This notebook loads all saved forward-model runs for the background-100k chains, rebuilds the test split, evaluates every best checkpoint, and assembles one shared figure across chains.

## What this notebook expects

- one directory per chain inside a common model root, for example `forward_background_100k/igh`, `forward_background_100k/igk`, ...
- each chain directory should contain `best.pt`, `history.json`, `test_metrics.json`, `data_stats.json`, `mean.npy`, `std.npy`
- AIRR tables and TCRemP parquet files must be reachable either through `data_stats.json` or through the path candidates configured below

If your actual paths differ, edit only the configuration cell.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from IPython.display import display

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

CHAINS = ['igh', 'igk', 'igl', 'tra', 'trb', 'trd', 'trg']
SPLIT_SEED = 42
TRAIN_FRACTION = 0.8
VAL_FRACTION = 0.1
EVAL_BATCH_SIZE = 512

MODEL_ROOT_CANDIDATES = [
    Path('/projects/immunestatus/vdjrearm/irrmcodec/forward_background_100k'),
    root / 'forward_background_100k',
    root / 'artifacts' / 'forward_background_100k',
]

AIRR_ROOT_CANDIDATES = [
    Path('/projects/immunestatus/vdjrearm/airr_format'),
    root / 'notebooks' / 'projects' / 'immunestatus' / 'vdjrearm' / 'airr_format',
]

EMBEDDINGS_ROOT_CANDIDATES = [
    Path('/projects/immunestatus/vdjrearm/tcremp'),
    root / 'notebooks' / 'projects' / 'immunestatus' / 'vdjrearm' / 'tcremp',
]

def first_existing(paths):
    for path in paths:
        if path is not None and Path(path).exists():
            return Path(path)
    return None

model_root = first_existing(MODEL_ROOT_CANDIDATES) or MODEL_ROOT_CANDIDATES[0]
airr_root = first_existing(AIRR_ROOT_CANDIDATES) or AIRR_ROOT_CANDIDATES[0]
embeddings_root = first_existing(EMBEDDINGS_ROOT_CANDIDATES) or EMBEDDINGS_ROOT_CANDIDATES[0]

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def embedding_candidates(chain):
    return [
        embeddings_root / f'{chain}_background_100k_tcremp.parquet',
        embeddings_root / f'{chain}_background_100k_embeddings.parquet',
        embeddings_root / f'{chain}_background_100k.parquet',
    ]

def resolve_chain_paths(chain):
    model_dir = model_root / chain
    stats = read_json(model_dir / 'data_stats.json') if (model_dir / 'data_stats.json').exists() else {}

    airr_candidates = []
    if stats.get('airr_path'):
        airr_candidates.append(Path(stats['airr_path']))
    airr_candidates.append(airr_root / f'{chain}_background_100k.tsv')

    emb_candidates = []
    if stats.get('embeddings_path'):
        emb_candidates.append(Path(stats['embeddings_path']))
    emb_candidates.extend(embedding_candidates(chain))

    airr_path = first_existing(airr_candidates) or airr_candidates[0]
    embeddings_path = first_existing(emb_candidates) or emb_candidates[0]

    return {
        'chain': chain,
        'model_dir': model_dir,
        'history_path': model_dir / 'history.json',
        'test_metrics_path': model_dir / 'test_metrics.json',
        'data_stats_path': model_dir / 'data_stats.json',
        'checkpoint_path': model_dir / 'best.pt',
        'mean_path': model_dir / 'mean.npy',
        'std_path': model_dir / 'std.npy',
        'airr_path': airr_path,
        'embeddings_path': embeddings_path,
    }

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_colwidth', 120)

print('root           :', root)
print('model_root     :', model_root)
print('airr_root      :', airr_root)
print('embeddings_root:', embeddings_root)

In [ ]:
path_rows = []
for chain in CHAINS:
    paths = resolve_chain_paths(chain)
    path_rows.append({
        'chain': chain,
        'model_dir': str(paths['model_dir']),
        'model_dir_exists': paths['model_dir'].exists(),
        'history_exists': paths['history_path'].exists(),
        'checkpoint_exists': paths['checkpoint_path'].exists(),
        'airr_path': str(paths['airr_path']),
        'airr_exists': paths['airr_path'].exists(),
        'embeddings_path': str(paths['embeddings_path']),
        'embeddings_exists': paths['embeddings_path'].exists(),
    })

paths_df = pd.DataFrame(path_rows)
display(paths_df)

available_chains = paths_df.loc[
    paths_df[['history_exists', 'checkpoint_exists', 'airr_exists', 'embeddings_exists']].all(axis=1),
    'chain',
].tolist()

if not available_chains:
    raise FileNotFoundError(
        'No complete chain configuration was found. Update the configuration cell so the notebook can see the model, AIRR, and parquet files.'
    )

print('Chains ready for evaluation:', ', '.join(available_chains))

In [ ]:
from irrm_codec.dataio import load_airr_with_embeddings
from irrm_codec.train_forward import build_dataloader, build_model, run_epoch
from irrm_codec.utils import apply_standardizer, choose_device, split_indices

def build_args_from_checkpoint(checkpoint, data_stats):
    extra = checkpoint.get('extra', {})
    return SimpleNamespace(
        max_len=extra.get('max_len', data_stats.get('max_len', 40)),
        hidden_dim=extra.get('hidden_dim', 192),
        mlp_dim=extra.get('mlp_dim', 512),
        mlp_hidden_dim=extra.get('mlp_hidden_dim', 1024),
        dropout=extra.get('dropout', 0.2),
        dilations=extra.get('dilations', '1,2,4,8'),
        encoder_type=extra.get('encoder_type', 'plain_conv'),
    )

def load_chain_result(chain):
    paths = resolve_chain_paths(chain)
    history = read_json(paths['history_path'])
    saved_test_metrics = read_json(paths['test_metrics_path'])
    data_stats = read_json(paths['data_stats_path'])

    df, emb, merge_stats = load_airr_with_embeddings(
        airr_path=paths['airr_path'],
        embeddings_path=paths['embeddings_path'],
        locus=chain,
        clone_id_col=data_stats.get('clone_id_column', 'clone_id'),
    )

    train_idx, val_idx, test_idx = split_indices(
        len(df),
        train_fraction=TRAIN_FRACTION,
        val_fraction=VAL_FRACTION,
        seed=SPLIT_SEED,
    )

    test_df = df.iloc[test_idx].reset_index(drop=True)
    mean = np.load(paths['mean_path'])
    std = np.load(paths['std_path'])
    test_emb = apply_standardizer(emb[test_idx], mean, std)

    device = choose_device()
    checkpoint = torch.load(paths['checkpoint_path'], map_location=device)
    args = build_args_from_checkpoint(checkpoint, data_stats)
    model = build_model(args, output_dim=test_emb.shape[1]).to(device)
    model.load_state_dict(checkpoint['model_state'])

    test_loader = build_dataloader(
        test_df,
        test_emb,
        batch_size=EVAL_BATCH_SIZE,
        max_len=args.max_len,
        shuffle=False,
        num_workers=0,
    )

    recomputed_test_metrics = run_epoch(
        model,
        test_loader,
        optimizer=None,
        device=device,
        stage='test',
        epoch=1,
        num_epochs=1,
        log_interval=0,
        show_progress=False,
    )

    sample_rows = []
    offset = 0
    model.eval()
    with torch.no_grad():
        for tokens, mask, target, _lengths in test_loader:
            batch_size = target.shape[0]
            tokens = tokens.to(device)
            mask = mask.to(device)
            target = target.to(device)
            pred = model(tokens, mask)

            cosine = F.cosine_similarity(pred, target, dim=-1).cpu().numpy()
            mse = ((pred - target) ** 2).mean(dim=-1).cpu().numpy()
            pred_norm = pred.norm(dim=-1).cpu().numpy()
            target_norm = target.norm(dim=-1).cpu().numpy()

            batch_lengths = test_df['junction_aa'].str.len().iloc[offset:offset + batch_size].to_numpy()
            offset += batch_size

            sample_rows.append(pd.DataFrame({
                'chain': chain,
                'raw_length': batch_lengths,
                'cosine': cosine,
                'mse': mse,
                'pred_norm': pred_norm,
                'target_norm': target_norm,
            }))

    sample_metrics = pd.concat(sample_rows, ignore_index=True)
    history_frame = pd.DataFrame({
        'epoch': [row['epoch'] for row in history],
        'train_loss': [row['train']['loss'] for row in history],
        'val_loss': [row['val']['loss'] for row in history],
        'train_mse': [row['train']['mse'] for row in history],
        'val_mse': [row['val']['mse'] for row in history],
        'train_cosine': [row['train']['cosine'] for row in history],
        'val_cosine': [row['val']['cosine'] for row in history],
    })

    return {
        'chain': chain,
        'paths': paths,
        'history': history,
        'history_frame': history_frame,
        'saved_test_metrics': saved_test_metrics,
        'recomputed_test_metrics': recomputed_test_metrics,
        'data_stats': data_stats,
        'merge_stats': merge_stats,
        'full_df': df,
        'test_df': test_df,
        'sample_metrics': sample_metrics,
    }

results = {}
summary_rows = []
for chain in available_chains:
    result = load_chain_result(chain)
    results[chain] = result

    saved_metrics = result['saved_test_metrics']
    eval_metrics = result['recomputed_test_metrics']
    hist = result['history_frame']
    per_length = result['sample_metrics'].groupby('raw_length', as_index=False).agg(
        mean_cosine=('cosine', 'mean'),
        mean_mse=('mse', 'mean'),
        n=('cosine', 'size'),
    )

    summary_rows.append({
        'chain': chain,
        'epochs_ran': len(hist),
        'best_val_loss': hist['val_loss'].min(),
        'final_train_loss': hist['train_loss'].iloc[-1],
        'saved_test_loss': saved_metrics['loss'],
        'eval_test_loss': eval_metrics['loss'],
        'saved_test_cosine': saved_metrics['cosine'],
        'eval_test_cosine': eval_metrics['cosine'],
        'test_loss_delta': abs(saved_metrics['loss'] - eval_metrics['loss']),
        'test_cosine_delta': abs(saved_metrics['cosine'] - eval_metrics['cosine']),
        'n_total': len(result['full_df']),
        'n_test': len(result['test_df']),
        'raw_len_min': result['full_df']['junction_aa'].str.len().min(),
        'raw_len_mean': result['full_df']['junction_aa'].str.len().mean(),
        'raw_len_max': result['full_df']['junction_aa'].str.len().max(),
        'best_length_mean_cosine': per_length['mean_cosine'].max(),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('chain').reset_index(drop=True)
display(summary_df.round(4))

In [ ]:
metric_heatmap = summary_df.set_index('chain')[[
    'best_val_loss',
    'eval_test_loss',
    'eval_test_cosine',
    'raw_len_mean',
]].copy()

fig, ax = plt.subplots(figsize=(8, max(4, 0.7 * len(metric_heatmap))))
sns.heatmap(metric_heatmap, annot=True, fmt='.3f', cmap='viridis', ax=ax)
ax.set_title('Cross-chain summary metrics')
plt.show()

fig, axes = plt.subplots(len(results), 4, figsize=(24, 4.5 * len(results)), constrained_layout=True)
if len(results) == 1:
    axes = np.array([axes])

for row_idx, chain in enumerate(sorted(results)):
    result = results[chain]
    hist = result['history_frame']
    lengths = result['full_df']['junction_aa'].str.len()
    samples = result['sample_metrics']
    test_metrics = result['recomputed_test_metrics']

    ax = axes[row_idx, 0]
    ax.plot(hist['epoch'], hist['train_loss'], marker='o', label='train')
    ax.plot(hist['epoch'], hist['val_loss'], marker='o', label='val')
    ax.axhline(test_metrics['loss'], color='black', linestyle='--', linewidth=1.5, label='test')
    ax.set_title(f'{chain.upper()}: loss')
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.legend(frameon=False, loc='best')

    ax = axes[row_idx, 1]
    ax.plot(hist['epoch'], hist['train_cosine'], marker='o', label='train')
    ax.plot(hist['epoch'], hist['val_cosine'], marker='o', label='val')
    ax.axhline(test_metrics['cosine'], color='black', linestyle='--', linewidth=1.5, label='test')
    ax.set_title(f'{chain.upper()}: cosine similarity')
    ax.set_xlabel('epoch')
    ax.set_ylabel('cosine')
    ax.legend(frameon=False, loc='best')

    ax = axes[row_idx, 2]
    bins = np.arange(lengths.min(), lengths.max() + 2) - 0.5
    ax.hist(lengths, bins=bins, color='#4C78A8', alpha=0.85, edgecolor='white')
    ax.axvline(lengths.mean(), color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'{chain.upper()}: raw CDR3 length distribution')
    ax.set_xlabel('length')
    ax.set_ylabel('count')

    ax = axes[row_idx, 3]
    ax.hist(samples['cosine'], bins=30, color='#F58518', alpha=0.85, edgecolor='white')
    ax.axvline(samples['cosine'].mean(), color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'{chain.upper()}: test cosine distribution')
    ax.set_xlabel('cosine')
    ax.set_ylabel('count')

fig.suptitle('IRRM-CODEC forward background-100k analysis across chains', fontsize=24, y=1.01)
plt.show()

per_length_fig, per_length_axes = plt.subplots(len(results), 1, figsize=(14, 3.5 * len(results)), constrained_layout=True)
if len(results) == 1:
    per_length_axes = [per_length_axes]

for ax, chain in zip(per_length_axes, sorted(results)):
    samples = results[chain]['sample_metrics']
    per_length = samples.groupby('raw_length', as_index=False).agg(
        mean_cosine=('cosine', 'mean'),
        n=('cosine', 'size'),
    )
    ax.plot(per_length['raw_length'], per_length['mean_cosine'], marker='o', color='#54A24B')
    ax.set_title(f'{chain.upper()}: mean test cosine by raw length')
    ax.set_xlabel('raw length')
    ax.set_ylabel('mean cosine')
    for x, y, n in per_length.itertuples(index=False):
        ax.text(x, y, str(n), fontsize=9, ha='center', va='bottom')

plt.show()